# World Cup 2026 Awards Predictions

This notebook uses simulated team paths as tournament exposure, directly simulates player outputs, estimates award probabilities, and recommends company-game picks using expected points.

## 1. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_URL = "https://github.com/wdqgallego-git/worldcup-predictor.git"
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path("/content/worldcup-predictor")
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
src_dir = PROJECT_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project directory: {PROJECT_DIR}")

## 2. Load team_path_simulations.csv

Run `01_match_predictions_colab.ipynb` or `python scripts/run_final_predictions.py` first. The award model uses simulated tournament paths as exposure and context.

In [ ]:
team_paths_path = OUTPUT_DIR / "team_path_simulations.csv"
champion_path = OUTPUT_DIR / "champion_probabilities.csv"
runner_up_path = OUTPUT_DIR / "runner_up_probabilities.csv"
required_team_outputs = [team_paths_path, champion_path, runner_up_path]
missing_team_outputs = [str(path) for path in required_team_outputs if not path.exists()]
if missing_team_outputs:
    raise FileNotFoundError(
        "Missing tournament outputs. Run the match-prediction pipeline first: "
        + ", ".join(missing_team_outputs)
    )

team_paths = pd.read_csv(team_paths_path)
champion_probabilities = pd.read_csv(champion_path)
runner_up_probabilities = pd.read_csv(runner_up_path)
print(f"Tournament simulations: {team_paths['simulation_id'].nunique():,}")
print(f"Team-path rows: {len(team_paths):,}")
team_paths.head()

## 3. Load player data

Real player files are preferred automatically. Sample files remain available as a development fallback until final squads and late injury adjustments are loaded.

In [ ]:
from player_data import (
    build_player_feature_table,
    load_goalkeeper_stats,
    load_injury_adjustments,
    load_penalty_takers,
    load_player_stats,
    load_players,
    load_squad_status,
)

players = load_players()
player_stats = load_player_stats(players=players)
goalkeeper_stats = load_goalkeeper_stats(players=players)
penalty_takers = load_penalty_takers(players=players)
squad_status = load_squad_status()
injury_adjustments = load_injury_adjustments()
player_features = build_player_feature_table(
    players=players,
    player_stats=player_stats,
    penalty_takers=penalty_takers,
    squad_status=squad_status,
    injury_adjustments=injury_adjustments,
)

print(f"Player features: {len(player_features):,}")
print(f"Goalkeepers: {len(goalkeeper_stats):,}")
player_features.head()

## 4. Run top scorer model

Open-play goals are simulated directly with a Negative Binomial model. Penalty goals and assists are simulated separately. Golden Boot ties are ranked by goals, assists, then fewer minutes.

In [ ]:
from awards_model import aggregate_award_probabilities, simulate_awards

award_results = simulate_awards(
    team_paths=team_paths,
    player_features=player_features,
    goalkeeper_stats=goalkeeper_stats,
    penalty_takers=penalty_takers,
    random_seed=2026,
)
award_probabilities = aggregate_award_probabilities(award_results, output_dir=OUTPUT_DIR)
top_scorer_probabilities = award_probabilities["top_scorer_probabilities"]
print(f"Simulated award tournaments: {len(award_results):,}")
top_scorer_probabilities.head(15)

## 5. Run MVP model

The MVP model uses a fixed transparent heuristic: team finish, attacking output, knockout impact, minutes, star prior, and a small position bonus.

In [ ]:
mvp_probabilities = award_probabilities["mvp_probabilities"]
mvp_probabilities.head(15)

## 6. Run Golden Glove model

The Golden Glove model uses team progression, clean sheets, save percentage, goals prevented, and goalkeeper reputation.

In [ ]:
golden_glove_probabilities = award_probabilities["golden_glove_probabilities"]
golden_glove_probabilities.head(15)

## 7. Calculate expected points

Each award is calculated independently. Runner-up uses `runner_up_probability` directly rather than finalist probability.

In [ ]:
from award_strategy import calculate_award_expected_points, load_scoring_rules

scoring_rules = load_scoring_rules()
probability_files = {
    "champion": (OUTPUT_DIR / "champion_probabilities.csv", "champion_probability"),
    "runner_up": (OUTPUT_DIR / "runner_up_probabilities.csv", "runner_up_probability"),
    "top_scorer": (OUTPUT_DIR / "top_scorer_probabilities.csv", "probability"),
    "mvp": (OUTPUT_DIR / "mvp_probabilities.csv", "probability"),
    "golden_glove": (OUTPUT_DIR / "golden_glove_probabilities.csv", "probability"),
}
award_expected_points = calculate_award_expected_points(
    scoring_rules=scoring_rules,
    probability_files=probability_files,
)
award_expected_points.groupby("category", sort=False).head(5)

## 8. Show safe and strategic picks

The safe pick maximizes expected points. A strategic alternative is shown only when it preserves at least 85–90% of the safe expected value.

In [ ]:
from award_strategy import build_award_picks

final_award_picks = build_award_picks(award_expected_points)
final_award_picks

## 9. Export awards output

In [ ]:
from award_strategy import save_award_strategy_outputs

strategy_outputs = save_award_strategy_outputs(
    expected_points=award_expected_points,
    final_picks=final_award_picks,
    output_dir=OUTPUT_DIR,
)
export_files = [
    "top_scorer_probabilities.csv",
    "mvp_probabilities.csv",
    "golden_glove_probabilities.csv",
    "award_expected_points.csv",
    "final_award_picks.csv",
    "award_strategy_recommendations.xlsx",
]
for file_name in export_files:
    path = OUTPUT_DIR / file_name
    print(f"{path}: {path.exists()}")

# Optional in Google Colab:
# from google.colab import files
# files.download(str(OUTPUT_DIR / "award_strategy_recommendations.xlsx"))